# 5 · Phasor Neural Networks

*Phasor networks, from the ground up — notebook 5 of 6.*

The same phase algebra trains by gradient descent. A `PhasorDense` layer applies a
complex linear map to phasors and renormalizes to the unit circle; stacking them
gives a phasor MLP that we train with `train()` and standard `Optimisers`.

We train once on a synthetic **bullseye** task, then run the *same trained weights*
two ways and compare their test accuracy:

- **atemporal** — the pure HD path: instantaneous complex linear maps (`loss_and_accuracy`);
- **oscillator** — the temporal path: the weights drive a bank of spiking oscillators
  over many cycles (`MakeSpiking` + `spiking_loss_and_accuracy`), and we watch the
  accuracy **converge** to the atemporal answer.

In [ ]:
using Pkg
Pkg.activate(joinpath(@__DIR__, ".."))
using PhasorNetworks
using Plots, Lux
using NNlib: tanh_fast
using Distributions: Normal
using OneHotArrays: onehotbatch, onecold
using Random: Xoshiro
using Statistics: mean

## Synthetic data

Two classes: an inner blob and an outer ring, in 2-D. Not linearly separable.

In [ ]:
function bullseye_data(n_s, rng)
    d = Normal(0.0, 0.08)
    y = rand(rng, (0, 1), n_s)
    r = rand(rng, d, n_s) .+ (0.4 .* y)
    phi = (rand(rng, Float64, n_s) .- 1) .* (2 * pi)
    data = Float32.(cat(r .* cos.(phi), r .* sin.(phi), dims=2)')
    labels = onehotbatch(y, 0:1)
    return data, labels
end

rng = Xoshiro(0)
xv, yv = bullseye_data(2000, rng)
scatter(xv[1, :], xv[2, :], group=vec(onecold(yv)), markersize=2, aspect_ratio=:equal,
        title="bullseye data", xlabel="x", ylabel="y")

## A phasor MLP

Real inputs are squashed into phases with `tanh`, then passed through two
`PhasorDense` layers. The identity (`x -> x`) slot is a placeholder we will swap for
a spiking encoder later, so the trainable layers keep the same positions and the
weights transfer with no retraining.

In [ ]:
model = Chain(x -> Phase.(tanh_fast.(x)),
              x -> x,                                                   # encoder slot
              PhasorDense(2 => 64, normalize_to_unit_circle, use_bias=true),
              x -> x,
              PhasorDense(64 => 2, normalize_to_unit_circle, use_bias=true))
ps, st = Lux.setup(rng, model)

loss(x, y, m, p, s) = mean(evaluate_loss(m(x, p, s)[1], y, :quadrature));

## Train

CPU backend, a few epochs over freshly-sampled batches.

In [ ]:
args = Args(batchsize = 64, epochs = 5, backend = :cpu)
train_loader = [bullseye_data(args.batchsize, rng) for _ in 1:100]
test_loader  = [bullseye_data(args.batchsize, rng) for _ in 1:20]

losses, ps, st = train(model, ps, st, train_loader, loss, args)
println("final training loss: ", round(losses[end], digits=4))
plot(losses, xlabel="step", ylabel="loss", label="", title="training loss")

## Run 1 — atemporal (pure HD)

The trained network as instantaneous phase algebra: each `PhasorDense` is a single
complex matrix multiply. This is the fast inference path; its test accuracy is the
target the oscillator must reach.

In [ ]:
_, acc_static = loss_and_accuracy(test_loader, model, ps, st, args, encoding=:quadrature)
println("atemporal (HD) test accuracy: ", round(acc_static, digits=4))

## Run 2 — oscillator (temporal)

Now run the **same weights** as spiking oscillators. `MakeSpiking` drops into the
encoder slot and converts the phase input into spike trains; the two `PhasorDense`
layers integrate as oscillator banks over `repeats` cycles. `ps`/`st` are reused
verbatim — no retraining. `spiking_loss_and_accuracy` reports accuracy **at each
cycle**, so we can watch it settle.

In [ ]:
repeats = 15
spk_args = SpikingArgs()

spk_model = Chain(x -> Phase.(tanh_fast.(x)),
                  MakeSpiking(spk_args, repeats),                       # encoder slot
                  PhasorDense(2 => 64, normalize_to_unit_circle, use_bias=true),
                  x -> x,
                  PhasorDense(64 => 2, normalize_to_unit_circle, use_bias=true))

spk_test = [bullseye_data(200, rng)]
_, acc_cycles = spiking_loss_and_accuracy(spk_test, spk_model, ps, st, args, encoding=:quadrature)
acc_cycles = vec(acc_cycles)
println("oscillator accuracy per cycle: ", round.(acc_cycles, digits=3))
println("oscillator (settled) accuracy: ", round(acc_cycles[end-1], digits=4))

## Convergence

The oscillator network starts near chance and **converges, cycle by cycle, to the
atemporal accuracy** (dashed line) as the banks settle into their phases. Both
mechanisms — pure HD and spiking oscillators — reach the same answer from the same
trained weights.

In [ ]:
plot(1:length(acc_cycles), acc_cycles, marker=:circle, label="oscillator (temporal)",
     xlabel="cycle", ylabel="test accuracy", title="oscillator accuracy converges to the HD answer",
     ylims=(0, 1), legend=:bottomright)
hline!([acc_static], label="atemporal (HD)", linestyle=:dash)

## Takeaway

A phasor network is an ordinary differentiable model whose activations are phases
and whose linear maps act on the unit circle. The very same trained weights run two
ways: instantaneously as pure HD operations, or over time as a bank of spiking
oscillators — and the temporal execution converges to the atemporal accuracy with
no retraining. The finale makes the *time* axis itself carry information.